In [3]:
import keras
(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data()
word_index = keras.datasets.imdb.get_word_index()  
id_to_word = {id_ + 3: word for word, id_ in word_index.items()} 
for id_, token in enumerate(("<pad>", "<sos>", "<unk>")): 
    id_to_word[id_] = token
" ".join([id_to_word[id_] for id_ in X_train[0][:10]]) 

1641221/1641221 [==============================] - 1s 1us/step


'<sos> this film was just brilliant casting location scenery story'

In [6]:
import tensorflow_datasets as tfds 
import tensorflow as tf
datasets, info = tfds.load("imdb_reviews", as_supervised=True, with_info=True) 
train_size = info.splits["train"].num_examples

def preprocess(X_batch, y_batch): 
    X_batch = tf.strings.substr(X_batch, 0, 300) 
    X_batch = tf.strings.regex_replace(X_batch, b"<br\\s / >", b" ") 
    X_batch = tf.strings.regex_replace(X_batch, b"[a-zA-Z']", b" ") 
    X_batch = tf.strings.split(X_batch) 
    return X_batch.to_tensor(default_value=b"<pad>"), y_batch 

from collections import Counter 
vocabulary = Counter() 
for X_batch, y_batch in datasets["train"].batch(32).map(preprocess): 
    for review in X_batch: 
        vocabulary.update(list(review.numpy())) 
vocabulary.most_common()[:3] 

[(b'<pad>', 210275), (b',', 56335), (b'.', 51604)]

In [7]:
vocab_size = 10000 
truncated_vocabulary = [ 
    word for word, count in vocabulary.most_common()[:vocab_size]]
words = tf.constant(truncated_vocabulary) 
word_ids = tf.range(len(truncated_vocabulary), dtype=tf.int64) 
vocab_init = tf.lookup.KeyValueTensorInitializer(words, word_ids) 
num_oov_buckets = 1000 
table = tf.lookup.StaticVocabularyTable(vocab_init, num_oov_buckets) 
table.lookup(tf.constant([b"This movie was faaaaaantastic".split()])) 

<tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[3371, 2971, 3263, 2679]], dtype=int64)>

In [9]:
def encode_words(X_batch, y_batch): 
    return table.lookup(X_batch), y_batch 
train_set = datasets["train"].batch(32).map(preprocess) 
train_set = train_set.map(encode_words).prefetch(1) 
embed_size = 128 
model = keras.models.Sequential([ 
    keras.layers.Embedding(vocab_size + num_oov_buckets, embed_size, 
                           input_shape=[None]), 
    keras.layers.GRU(128, return_sequences=True), 
    keras.layers.GRU(128), 
    keras.layers.Dense(1, activation="sigmoid") 
]) 
model.compile(loss="binary_crossentropy", optimizer="adam", 
              metrics=["accuracy"]) 
history = model.fit(train_set, epochs=5) 


Epoch 1/5



782/782 [==============================] - 12s 11ms/step - loss: 0.6906 - accuracy: 0.5266
Epoch 2/5
782/782 [==============================] - 8s 11ms/step - loss: 0.6814 - accuracy: 0.5592
Epoch 3/5
782/782 [==============================] - 8s 11ms/step - loss: 0.6698 - accuracy: 0.5814
Epoch 4/5
782/782 [==============================] - 8s 11ms/step - loss: 0.6518 - accuracy: 0.5975
Epoch 5/5
782/782 [==============================] - 8s 11ms/step - loss: 0.6349 - accuracy: 0.6045


In [10]:
#处理掩码
K = keras.backend 
inputs = keras.layers.Input(shape=[None]) 
mask = keras.layers.Lambda(lambda inputs: K.not_equal(inputs, 0))(inputs) 
z = keras.layers.Embedding(vocab_size + num_oov_buckets, embed_size)(inputs) 
z = keras.layers.GRU(128, return_sequences=True)(z, mask=mask) 
z = keras.layers.GRU(128)(z, mask=mask) 
outputs = keras.layers.Dense(1, activation="sigmoid")(z) 
model = keras.Model(inputs=[inputs], outputs=[outputs]) 


In [ ]:
#在情感分析模型中使用nnlm-en-dim50句子嵌入模块
import tensorflow_hub as hub 
model = keras.Sequential([ 
    hub.KerasLayer("https://tfhub.dev/google/tf2-preview/nnlm-en-dim50/1", 
                   dtype=tf.string, input_shape=[], output_shape=[50]), 
    keras.layers.Dense(128, activation="relu"), 
    keras.layers.Dense(1, activation="sigmoid") 
]) 
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"]) 

datasets, info = tfds.load("imdb_reviews", as_supervised=True, with_info=True) 
train_size = info.splits["train"].num_examples 
batch_size = 32 
train_set = datasets["train"].batch(batch_size).prefetch(1) 
history = model.fit(train_set, epochs=5)